In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoConfig, AutoModel, DataCollatorWithPadding

# ==========================================
# 1. SET LOCAL FOLDER PATHS & DEVICE (CPU OR GPU)
# ==========================================
BASE_DIR = os.getcwd()
VALID_FILE = os.path.join(BASE_DIR, "val.jsonl")
TEST_FILE  = os.path.join(BASE_DIR, "test.jsonl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running locally on device: {device}")
print(f"Working directory: {BASE_DIR}")

variants = [
    ("Model 1 (seed=42, lr=2.0e-5)",   "submission_base_seed42_lr2e5"),
    ("Model 2 (seed=100, lr=1.5e-5)",  "submission_base_seed100_lr15e5"),
    ("Model 3 (seed=2026, lr=2.5e-5)", "submission_base_seed2026_lr25e5"),
]

label_map = {"not_entailment": 0, "entailment": 1}
id2label = {0: "not_entailment", 1: "entailment"}

# ==========================================
# 2. STEP 1: LOAD & DISPLAY VALIDATION SCORES FIRST
# ==========================================
if os.path.exists(VALID_FILE):
    val_df = pd.read_json(VALID_FILE, lines=True)
    y_val_true = val_df["label"].map(label_map).values

    val_rows = []
    for display_name, folder in variants:
        pred_path = os.path.join(BASE_DIR, folder, "valid_predictions.jsonl")
        if os.path.exists(pred_path):
            vp_df = pd.read_json(pred_path, lines=True)
            y_val_pred = vp_df["label"].map(label_map).values
            val_rows.append({
                "Model Variant": display_name,
                "Valid Accuracy": round(accuracy_score(y_val_true, y_val_pred), 4),
                "Valid F1": round(f1_score(y_val_true, y_val_pred, average="binary"), 4),
            })

    print("\n============================================================")
    print("STEP 1: SAVED VALIDATION SCORES (BEFORE RUNNING TEST DATA)")
    print("============================================================")
    print(pd.DataFrame(val_rows).to_string(index=False))
else:
    print(f"\nNote: 'val.jsonl' not found in {BASE_DIR}; skipping Step 1 table.")

# ==========================================
# 3. DEFINE MODEL ARCHITECTURE (CPU-COMPATIBLE FLOAT32)
# ==========================================
class DebertaV3EntailmentMSD(nn.Module):
    def __init__(self, model_name_or_dir, num_labels=2, offline=False):
        super().__init__()
        if offline and os.path.exists(model_name_or_dir):
            self.config = AutoConfig.from_pretrained(model_name_or_dir, local_files_only=True)
            self.deberta = AutoModel.from_config(self.config, attn_implementation="eager")
        else:
            self.config = AutoConfig.from_pretrained(model_name_or_dir)
            self.config.update({"hidden_dropout_prob": 0.1, "attention_probs_dropout_prob": 0.1})
            self.deberta = AutoModel.from_pretrained(
                model_name_or_dir, config=self.config, attn_implementation="eager", dtype=torch.float32
            )
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        pooled = torch.sum(last_hidden * mask, 1) / mask.sum(1).clamp(min=1e-9)
        return sum([self.classifier(drop(pooled)) for drop in self.dropouts]) / len(self.dropouts)

# ==========================================
# 4. STEP 2: LOAD ALL 3 SAVED MODELS & EVALUATE ON TEST.JSONL
# ==========================================
if not os.path.exists(TEST_FILE):
    print(f"\nSetup verified! Place 'test.jsonl' in {BASE_DIR} on demo day and re-run this cell.")
else:
    tok_dir = os.path.join(BASE_DIR, "saved_tokenizer")
    cfg_dir = os.path.join(BASE_DIR, "saved_config")
    use_offline = os.path.exists(tok_dir) and os.path.exists(cfg_dir)

    if use_offline:
        tokenizer = AutoTokenizer.from_pretrained(tok_dir, use_fast=False, local_files_only=True)
        model_source = cfg_dir
    else:
        tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base", use_fast=False)
        model_source = "microsoft/deberta-v3-base"

    test_df = pd.read_json(TEST_FILE, lines=True)
    has_labels = "label" in test_df.columns and test_df["label"].notna().all()

    if has_labels:
        y_test_true = test_df["label"].map(label_map).values
        y_test_tensor = torch.tensor(y_test_true, dtype=torch.long)

    test_ds = Dataset.from_pandas(test_df[["premise", "hypothesis"]]).map(
        lambda b: tokenizer(b["premise"], b["hypothesis"], truncation="only_first", max_length=384),
        batched=True,
        remove_columns=["premise", "hypothesis"],
    )
    loader = DataLoader(test_ds, batch_size=8, collate_fn=DataCollatorWithPadding(tokenizer))

    out_dir = os.path.join(BASE_DIR, "test_predictions")
    os.makedirs(out_dir, exist_ok=True)
    test_rows = []

    print("\n============================================================")
    print("STEP 2: RUNNING ALL 3 SAVED MODELS ON TEST.JSONL")
    print("============================================================")

    for display_name, folder in variants:
        weights_path = os.path.join(BASE_DIR, folder, "best_model_weights.pt")
        model = DebertaV3EntailmentMSD(model_source, num_labels=2, offline=use_offline)

        state_dict = torch.load(weights_path, map_location=device)
        model.load_state_dict(state_dict)
        model.float()
        model.to(device)
        model.eval()

        logits_list = []
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                logits_list.append(model(**batch).logits.cpu())

        full_logits = torch.cat(logits_list, dim=0)
        pred_ids = torch.argmax(full_logits, dim=-1).numpy()

        pred_file = os.path.join(out_dir, f"{folder}_test_predictions.jsonl")
        with open(pred_file, "w") as f:
            for idx, p_id in zip(test_df["idx"], pred_ids):
                f.write(json.dumps({"idx": int(idx), "label": id2label[int(p_id)]}) + "\n")

        if has_labels:
            t_loss = F.cross_entropy(full_logits, y_test_tensor).item()
            t_acc = accuracy_score(y_test_true, pred_ids)
            t_f1 = f1_score(y_test_true, pred_ids, average="binary")
            test_rows.append({
                "Model Variant": display_name,
                "Test Loss": round(t_loss, 4),
                "Test Accuracy": round(t_acc, 4),
                "Test F1": round(t_f1, 4),
                "Saved Predictions": f"test_predictions/{folder}_test_predictions.jsonl",
            })
        else:
            test_rows.append({
                "Model Variant": display_name,
                "Test Loss": "Hidden",
                "Test Accuracy": "Hidden",
                "Test F1": "Hidden",
                "Saved Predictions": f"test_predictions/{folder}_test_predictions.jsonl",
            })

    res_df = pd.DataFrame(test_rows)
    print(res_df.to_string(index=False))

    if has_labels:
        best = res_df.sort_values(by="Test Accuracy", ascending=False).iloc[0]
        print(f"\n>>> BEST MODEL: {best['Model Variant']} | Test Accuracy: {best['Test Accuracy']} | Test Loss: {best['Test Loss']} <<<")
